In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 12, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 12, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

FETCHING INTRADAY SDR SLICES...: 100%|██████████| 18/18 [00:00<00:00, 32.85it/s]


In [3]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
sdf

Classifying Trades: 100%|██████████| 590/590 [00:00<00:00, 1318.67trade/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
0,NEWT-NOVA,1690122584000000201,2026-01-12 05:03:52+00:00,2026-01-07,2026-01-12,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT 4Dx10Y RECEI...,200000.0,USD,False,...,NA/Swap OIS USD,QZJ92TTHTSF0,BILT,N,False,NaN,0.0,NaN,None,NaN
1,NEWT-NOVA,1690122583000000101,2026-01-12 05:04:29+00:00,2026-01-07,2026-01-12,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 4Dx10Y PAYER...,700000.0,USD,False,...,NA/Swap OIS USD,QZZLNQ2D4JQT,BILT,N,False,NaN,0.0,NaN,None,NaN
2,NEWT-NOVA,1690096990000000101,2026-01-12 05:10:38+00:00,2026-01-07,2026-01-12,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT 4Dx10Y RECEI...,400000.0,USD,False,...,NA/Swap OIS USD,QZJ92TTHTSF0,BILT,N,False,NaN,0,NaN,None,NaN
3,NEWT-NOVA,1690097663000000101,2026-01-12 05:11:55+00:00,2026-01-07,2026-01-12,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 4Dx10Y PAYER...,1000000.0,USD,False,...,NA/Swap OIS USD,QZZLNQ2D4JQT,BILT,N,False,NaN,0,NaN,None,NaN
4,NEWT-NOVA,1690101567000000101,2026-01-12 05:15:49+00:00,2026-01-07,2026-01-12,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1Y CONSTANT 4Dx10Y RECEI...,200000.0,USD,False,...,NA/Swap OIS USD,QZJ92TTHTSF0,BILT,N,False,NaN,0.0,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
529,NEWT-TRAD / MODI-TRAD,1698615334000000201 / 1698642209000000301,2026-01-12 19:12:40+00:00,2026-01-12,2026-01-20,SWAPTION_RECEIVER / SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 1Wx10Y RECEI...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZMMWR8JKZQ8 / QZWXKVHB5F8V,BGCD,N,True,"260,000","260,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2.0
530,MODI-TRAD,1698615338000000601 / 1698630753000000201,2026-01-12 19:12:47+00:00,2026-01-12,2026-01-13,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Dx10Y PAYER...,50000000.0,USD,False,...,NA/Swap Fxd Flt USD,QZWXKVHB5F8V / QZMMWR8JKZQ8,BGCD,N,True,"135,000","135,000",1.0,platform=BGCD; time_delta_max=0.0s; premium_mo...,2.0
531,NEWT-NOVA,1698657380000000201 / 1698711263000000201,2026-01-12 19:14:39+00:00,2024-11-25,2029-12-19,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT IMM_Z2029xIM...,80000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,BILT,N,True,"11,296,000","11,296,000",1.0,platform=BILT; time_delta_max=0.0s; premium_mo...,2.0
532,NEWT-NOVA,1698656467000000101 / 1698697675000000201,2026-01-12 19:14:45+00:00,2024-11-26,2029-12-19,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT IMM_Z2029xIM...,69000000.0,USD,False,...,NA/Swap OIS USD,QZZGWPNBF5R3 / QZNLQ8T0N0SX,BILT,N,True,"9,742,800","9,742,800",1.0,platform=BILT; time_delta_max=0.0s; premium_mo...,2.0


In [ ]:
# temp = sdf.copy()
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp[temp["package_type"] == "SWAPTION"].to_excel("filter_swaptions_trades_no_pkg.xlsx")
# sdf["trade_label"].value_counts()


# sdf["package_type"].value_counts()
sdf[~sdf["trade_label"].str.contains("4Dx10Y")]["package_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & ((sdf["forward_label"] == "3M")) & ((sdf["tenor_label"] == "10Y"))]
# sdf[sdf["trade_label"].str.contains("3Mx10Y")]

package_type
SWAPTION                 193
STRADDLE                  42
VERTICAL_SPREAD_1x1        4
VERTICAL_SPREAD_1x1.5      2
VERTICAL_SPREAD_1x2        2
Name: count, dtype: int64

In [41]:
# ["platform_identifier"].value_counts()
sdf = sdf[~(sdf["trade_label"].str.contains("4Dx10Y")) & (sdf["package_type"] == "SWAPTION") & ~(sdf["platform_identifier"].isin(["BILT", "XXXX"]))]
sdf["execution_timestamp"] = sdf["execution_timestamp"].astype(str)
sdf.to_excel("temp.xlsx")

C:\Users\chris\AppData\Local\Temp\ipykernel_81588\1880684314.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [18]:
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf[ "package_indicator"] == False)]
# sdf["package_type"].value_counts()
# sdf[(sdf["package_type"] == "VERTICAL_SPREAD_1x1")]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count


In [33]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x000001E1A5506850> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x000001E1DD3B6670> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 7, 0, 0)})

In [45]:
from SDRUtils.products._swaptions.pricer import usd_swaption_straddle_pricer_from_row, usd_swaption_leg_pricer_from_row

# sdf.loc[285], sdf.loc[286]
# .iloc[0].to_dict()
# df[df["Original Dissemination Identifier"] == 1653435994000001301]
usd_swaption_leg_pricer_from_row(sdf.loc[77], pricer, leg="receiver", fwd_prem=87850)
# usd_swaption_leg_pricer_from_row(sdf.loc[15], pricer, leg="receiver", fwd_prem=565000.0)
# usd_swaption_straddle_pricer_from_row(sdf.loc[374], pricer)

USDSwaptionLegPricerResult(ql_swaption=<QuantLib.QuantLib.Swaption; proxy of <Swig Object of type 'ext::shared_ptr< Swaption > *' at 0x000001E1DE14F3C0> >, iv_bpvol_yr=59.88234499377047, dv01=-6000.683971513071, vega01=1411.2991555498484, gamma01=58.45398077769685, theta1d=1290.3227395978174)

In [35]:
# sdf.loc[336]["package_reason"]
sdf.loc[4]

event_action                                                          NEWT-NOVA
trade_id                                                    1650906616000000701
execution_timestamp                                   2026-01-07 12:48:47+00:00
effective_date                                              2026-01-02 00:00:00
expiration_date                                             2026-04-02 00:00:00
product_type                                                  SWAPTION_RECEIVER
trade_label                   USD-SOFR-OIS Compound 1Y CONSTANT 3Mx10Y RECEI...
notional                                                             50000000.0
notional_currency                                                           USD
is_notional_capped                                                        False
estimated_pv01                                                              0.0
package_type                                                           SWAPTION
package_id                              

In [46]:
4.09* np.sqrt(252)

np.float64(64.92673717352505)

In [9]:
ids = [
1698486377000000201,
1698641398000000301,
1698628143000000201,
1698635498000000401,
1698635499000000501,

]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("temp_trades.csv",index=False)

sdf[sdf["trade_id"].isin([str(id) for id in ids])].to_dict(orient="records")

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'event_action': 'NEWT-TRAD',
  'trade_id': '1698486377000000201',
  'execution_timestamp': Timestamp('2026-01-12 18:34:37+0000', tz='UTC'),
  'effective_date': Timestamp('2026-01-12 00:00:00'),
  'expiration_date': Timestamp('2026-07-13 00:00:00'),
  'product_type': 'SWAPTION_RECEIVER',
  'trade_label': 'USD-SOFR-OIS Compound 1D CONSTANT 6Mx5Y RECEIVER EURO VANILLA PHYS',
  'notional': 350000000.0,
  'notional_currency': 'USD',
  'is_notional_capped': False,
  'estimated_pv01': 0.0,
  'package_type': 'SWAPTION',
  'package_id': None,
  'package_legs': None,
  'underlying_expiration_date': Timestamp('2031-07-15 00:00:00'),
  'tenor_years': 5.008219178082192,
  'tenor_label': '5Y',
  'forward_start_years': 0.4986301369863014,
  'forward_label': '6M',
  'premium': 6510000.0,
  'exercise_style': 'EUROPEAN',
  'strike': 0.03509,
  'upi_underlier_name': 'NA/Swap OIS USD',
  'unique_product_identifier': 'QZNLQ8T0N0SX',
  'platform_identifier': 'TPSE',
  'cleared': 'N',
  'package_indicator'